### Step 1: Install necesscary packages

In [ ]:
!pip install matplotlib
!pip install torch numpy transformers datasets tiktoken wandb tqdm

### Step 2: Package imports and configuration

In [2]:
import sys
import os
sys.path.append(os.path.abspath("..")) 
os.environ["CUDA_VISIBLE_DEVICES"] = "1"
import torch
import torch.nn as nn
import torch.nn.functional as F
import random
import pickle
from model import GPT, GPTConfig
import random
from tqdm import tqdm
import time
import json
import matplotlib.pyplot as plt
# Configuration
beta = 0.5
device = 'cuda' if torch.cuda.is_available() else 'cpu'
base_lr = 1e-4
epochs = 5
batch_size = 64
max_length =64
num_samples = 1
max_new_tokens = 200
temperature = 0.8
top_k = 200
# tokenizer
with open("../sft/meta.pkl", "rb") as f:
    meta = pickle.load(f)
stoi, itos = meta["stoi"], meta["itos"]
def encode(s): return [stoi[c] for c in s]
def decode(l): return ''.join([itos[i] for i in l])

### Step 3: Define helper functions

In [3]:
def compute_logprob(input_ids):
    inputs = input_ids[:, :-1]
    targets = input_ids[:, 1:]
    logits, _ = gpt(inputs, full_seq=True)
    B, T, V = logits.size()
    logits_flat = logits.reshape(-1, V)
    targets_flat = targets.reshape(-1)
    loss = F.cross_entropy(logits_flat, targets_flat, ignore_index=0, reduction='none')
    loss = loss.reshape(B, T)
    attention_mask = (targets != 0).float()
    loss = (loss * attention_mask).sum(dim=1) / attention_mask.sum(dim=1)
    return -loss 

def pad_or_truncate(seq, max_length):
    return seq[-max_length:] if len(seq) > max_length else seq + [0] * (max_length - len(seq))

def get_batches(lines, batch_size):
    random.shuffle(lines)
    #for l in lines:
    #    print(l[1])
    for i in range(0, len(lines), batch_size):
        batch = lines[i:i+batch_size]
        if len(batch) < batch_size:
            continue
        neg_inputs = [pad_or_truncate(encode(p['negative'] + '\n\n\n\n'), max_length) for p in batch]
        pos_inputs = [pad_or_truncate(encode(p['positive'] + '\n\n\n\n'), max_length) for p in batch]
        neg_tensor = torch.tensor(neg_inputs, dtype=torch.long, device=device)
        pos_tensor = torch.tensor(pos_inputs, dtype=torch.long, device=device)
        yield neg_tensor, pos_tensor

### Step 4: Load the pretrained NanoGPT model

In [ ]:
ckpt = torch.load("../sft/gpt.pt", map_location=device)
gptconf = GPTConfig(**ckpt['model_args'])
gpt = GPT(gptconf)
state_dict = ckpt['model']
unwanted_prefix = '_orig_mod.'
for k in list(state_dict.keys()):
    if k.startswith(unwanted_prefix):
        state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)
gpt.load_state_dict(state_dict)
gpt.to(device).train()

### Step 5: Load Data (**students are required to complete this part!**)

In [ ]:
# Step 5: load and validate preference pairs.
# The supplied pos_neg_pairs.json has only four DEMO examples.
# Replace/extend it with a larger, checked preference dataset for full training.
from pathlib import Path

data_path = Path("pos_neg_pairs.json")
if not data_path.exists():
    data_path = Path("data/pos_neg_pairs.json")
if not data_path.exists():
    raise FileNotFoundError("Missing dpo/pos_neg_pairs.json (run this notebook from dpo/)")

with data_path.open("r", encoding="utf-8") as f:
    raw_pairs = json.load(f)
if not isinstance(raw_pairs, list) or not raw_pairs:
    raise ValueError("Expected a nonempty JSON array of positive/negative pairs")

lines = []
for index, pair in enumerate(raw_pairs):
    if not isinstance(pair, dict) or not all(k in pair for k in ("positive", "negative")):
        raise ValueError(f"Sample {index} must contain positive and negative strings")
    pos, neg = pair["positive"], pair["negative"]
    if not isinstance(pos, str) or not isinstance(neg, str) or pos == neg:
        raise ValueError(f"Invalid responses at sample {index}")

    # The question ends with '=?' in the course's arithmetic/algebra format.
    # This is more reliable than finding the longest common prefix: two numeric
    # answers can share initial digits, which would mask away answer tokens.
    end = pos.find("=?")
    if end < 0:
        raise ValueError(f"Sample {index}: missing '=?' in positive response")
    prompt = pos[:end + 2]
    if not neg.startswith(prompt):
        raise ValueError(f"Sample {index}: positive/negative prompts do not match")
    for response in (pos, neg):
        missing = sorted(set(response) - set(stoi))
        if missing:
            raise ValueError(f"Sample {index} contains unknown characters: {missing}")
    lines.append({"positive": pos, "negative": neg, "prompt": prompt})

max_pair_length = max(max(len(pair["positive"]), len(pair["negative"])) for pair in lines)
if max_pair_length > gpt.config.block_size:
    raise ValueError(
        f"Longest training response ({max_pair_length}) exceeds "
        f"model block_size ({gpt.config.block_size})"
    )
max_length = max(max_length, max_pair_length)  # Never truncate away the question.
if len(lines) < batch_size:
    print(f"Only {len(lines)} pairs: shrinking batch_size for a pipeline smoke test.")
    batch_size = len(lines)
print(f"Loaded {len(lines)} preference pairs from {data_path}")
print(f"batch_size={batch_size}, max_length={max_length}")
if len(lines) <= 4:
    print("WARNING: These are the four course examples, NOT a real training dataset.")


### Step 6: Build the optimizer and scheduler (**students are required to complete this part!**)

In [ ]:
# Step 6: optimize the trainable NanoGPT policy (the reference stays frozen).
import math

optimizer = torch.optim.AdamW(gpt.parameters(), lr=base_lr, weight_decay=0.01)
steps_per_epoch = math.ceil(len(lines) / batch_size)
total_steps = epochs * steps_per_epoch
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=total_steps, eta_min=base_lr * 0.1
)
print(f"AdamW lr={base_lr}, epochs={epochs}, total optimizer steps={total_steps}")


### Step 7: Begin training (**students are required to complete this part!**)

In [ ]:
# Step 7: standard DPO training from the provided SFT checkpoint.
# L_DPO = -log sigmoid(beta * [(log pi(pos|prompt) - log pi(neg|prompt))
#                             - (log ref(pos|prompt) - log ref(neg|prompt))])
# The frozen reference is a copy of the original model. No SFT is performed.
import copy

random.seed(42)
torch.manual_seed(42)
reference_gpt = copy.deepcopy(gpt).to(device).eval()
for parameter in reference_gpt.parameters():
    parameter.requires_grad_(False)


def make_dpo_batch(batch):
    """Right-pad samples; return tensors and shared prompt lengths."""
    longest = max(
        max(len(item["positive"]), len(item["negative"])) for item in batch
    )
    if longest > gpt.config.block_size:
        raise ValueError("Sample longer than model context window")

    def pack(key):
        encoded = [encode(item[key]) for item in batch]
        return torch.tensor(
            [tokens + [0] * (longest - len(tokens)) for tokens in encoded],
            dtype=torch.long, device=device,
        )

    prompt_lengths = torch.tensor(
        [len(encode(item["prompt"])) for item in batch],
        dtype=torch.long, device=device,
    )
    return pack("positive"), pack("negative"), prompt_lengths


def completion_logprob(model, ids, prompt_lengths):
    """Sum log p(response tokens | question), excluding prompt and padding."""
    # logits[:, t] predicts the character ids[:, t+1].
    logits, _ = model(ids[:, :-1], full_seq=True)
    targets = ids[:, 1:]
    token_logp = F.log_softmax(logits, dim=-1).gather(
        dim=-1, index=targets.unsqueeze(-1)
    ).squeeze(-1)

    # Positions refer to indices in the *original* (unshifted) sequence.
    positions = torch.arange(1, ids.shape[1], device=ids.device)[None, :]
    is_completion = positions >= prompt_lengths[:, None]
    valid = is_completion & (targets != 0)  # ID 0 is also the pad token.
    if not valid.any(dim=1).all():
        raise ValueError("A sample has an empty completion")
    return (token_logp * valid).sum(dim=1)


gpt.train()
for epoch in range(epochs):
    shuffled = list(lines)
    random.shuffle(shuffled)
    epoch_loss = 0.0
    preference_count = 0
    seen = 0
    pbar = tqdm(range(0, len(shuffled), batch_size), desc=f"Epoch {epoch+1}/{epochs}")

    for start in pbar:
        batch = shuffled[start:start + batch_size]
        pos_ids, neg_ids, prompt_lengths = make_dpo_batch(batch)

        # Trainable policy.
        pos_logprob = completion_logprob(gpt, pos_ids, prompt_lengths)
        neg_logprob = completion_logprob(gpt, neg_ids, prompt_lengths)

        # Reference model is frozen at the initial checkpoint.
        with torch.no_grad():
            ref_pos_logprob = completion_logprob(
                reference_gpt, pos_ids, prompt_lengths
            )
            ref_neg_logprob = completion_logprob(
                reference_gpt, neg_ids, prompt_lengths
            )

        policy_logratio = pos_logprob - neg_logprob
        reference_logratio = ref_pos_logprob - ref_neg_logprob
        loss = -F.logsigmoid(beta * (policy_logratio - reference_logratio)).mean()

        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(gpt.parameters(), max_norm=1.0)
        optimizer.step()
        scheduler.step()

        n = len(batch)
        seen += n
        epoch_loss += loss.item() * n
        preference_count += (policy_logratio.detach() > 0).sum().item()
        pbar.set_postfix(
            loss=f"{loss.item():.4f}",
            chosen_preference=f"{preference_count / seen:.1%}",
        )

    print(
        f"Epoch {epoch+1}: mean DPO loss={epoch_loss/seen:.4f}, "
        f"chosen preference rate={preference_count/seen:.1%}"
    )
    ckpt_path = "./dpo.pt"
    torch.save({
        "model_state_dict": gpt.state_dict(),
        "model_args": ckpt["model_args"],
    }, ckpt_path)
    print(f"Saved checkpoint to {ckpt_path}")

# Preference rate is NOT math accuracy; evaluate held-out questions separately.


### Step 8: Begin testing (**students are required to complete this part!**)

In [ ]:
# Load the fine-tuned model
ckpt_path = "../dpo/dpo.pt"
checkpoint = torch.load(ckpt_path, map_location=device)
gptconf = GPTConfig(**checkpoint['model_args'])
gpt = GPT(gptconf).cuda()
try:
    state_dict = checkpoint['model']
except:
    state_dict = checkpoint['model_state_dict']
unwanted_prefix = '_orig_mod.'
for k,v in list(state_dict.items()):
    if k.startswith(unwanted_prefix):
        state_dict[k[len(unwanted_prefix):]] = state_dict.pop(k)
gpt.load_state_dict(state_dict)
# Test
gpt.eval()
test_set = ["17+19=?", "3*17=?", "72/4=?", "72-x=34,x=?", "x*11=44,x=?", "3*17=?", "72/4=?", "72-x=34,x=?"]
with torch.no_grad():
    for prompt in test_set: 
        prompt_ids = encode(prompt)
        ###########################################################
        # Please complete the test code here!
        # ...
        # gpt.generate(x, max_new_tokens, temperature=temperature, top_k=top_k)
        # ...
        ###########################################################